# Dataset Loading & Schema Analysis

Dataset:
CICIDS2017 (MachineLearningCVE)

Goal:
- Load all raw datasets
- Verify schema consistency
- Analyze dataset sizes
- Build a data dictionary
- Document initial observations

In [1]:
from pathlib import Path

import pandas as pd
import polars as pl
import numpy as np

In [2]:
DATA_DIR = Path("../data/raw")

csv_files = sorted(DATA_DIR.glob("*.csv"))

print(f"Found {len(csv_files)} files:\n")

for file in csv_files:
    print(file.name)

Found 8 files:

Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
Friday-WorkingHours-Morning.pcap_ISCX.csv
Monday-WorkingHours.pcap_ISCX.csv
Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv
Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv
Tuesday-WorkingHours.pcap_ISCX.csv
Wednesday-workingHours.pcap_ISCX.csv


In [3]:
sample_df = pd.read_csv(csv_files[0])

sample_df.head()

,Destination Port,Flow Duration,Total Fwd Packets,Total Backward Packets,Total Length of Fwd Packets,Total Length of Bwd Packets,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,...,min_seg_size_forward,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label
0,54865,3,2,0,12,0,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
1,55054,109,1,1,6,6,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
2,55055,52,1,1,6,6,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
3,46236,34,1,1,6,6,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
4,54863,3,2,0,12,0,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN


In [4]:
print("Rows:", sample_df.shape[0])
print("Columns:", sample_df.shape[1])

Rows: 225745
Columns: 79


In [5]:
sample_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 225745 entries, 0 to 225744
Data columns (total 79 columns):
 #   Column                        Non-Null Count   Dtype  
---  ------                        --------------   -----  
 0    Destination Port             225745 non-null  int64  
 1    Flow Duration                225745 non-null  int64  
 2    Total Fwd Packets            225745 non-null  int64  
 3    Total Backward Packets       225745 non-null  int64  
 4   Total Length of Fwd Packets   225745 non-null  int64  
 5    Total Length of Bwd Packets  225745 non-null  int64  
 6    Fwd Packet Length Max        225745 non-null  int64  
 7    Fwd Packet Length Min        225745 non-null  int64  
 8    Fwd Packet Length Mean       225745 non-null  float64
 9    Fwd Packet Length Std        225745 non-null  float64
 10  Bwd Packet Length Max         225745 non-null  int64  
 11   Bwd Packet Length Min        225745 non-null  int64  
 12   Bwd Packet Length Mean       225745 non-null  float64


In [6]:
memory_mb = sample_df.memory_usage(deep=True).sum() / 1024**2

print(f"Memory Usage: {memory_mb:.2f} MB")

Memory Usage: 145.94 MB


In [7]:
schemas = {}

for file in csv_files:
    df = pd.read_csv(file, nrows=5)
    schemas[file.name] = list(df.columns)

In [8]:
reference = schemas[csv_files[0].name]

for name, columns in schemas.items():
    print(
        f"{name}:",
        columns == reference
    )

Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv: True
Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv: True
Friday-WorkingHours-Morning.pcap_ISCX.csv: True
Monday-WorkingHours.pcap_ISCX.csv: True
Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv: True
Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv: True
Tuesday-WorkingHours.pcap_ISCX.csv: True
Wednesday-workingHours.pcap_ISCX.csv: True


In [9]:
summary = []

for file in csv_files:
    df = pd.read_csv(file)

    summary.append({
        "file": file.name,
        "rows": df.shape[0],
        "columns": df.shape[1]
    })

summary_df = pd.DataFrame(summary)

summary_df

,file,rows,columns
0,Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv,225745,79
1,Friday-WorkingHours-Afternoon-PortScan.pcap_IS...,286467,79
2,Friday-WorkingHours-Morning.pcap_ISCX.csv,191033,79
3,Monday-WorkingHours.pcap_ISCX.csv,529918,79
4,Thursday-WorkingHours-Afternoon-Infilteration....,288602,79
5,Thursday-WorkingHours-Morning-WebAttacks.pcap_...,170366,79
6,Tuesday-WorkingHours.pcap_ISCX.csv,445909,79
7,Wednesday-workingHours.pcap_ISCX.csv,692703,79


In [11]:
summary_df["rows"].sum()

np.int64(2830743)

In [12]:
for file in csv_files:
    df = pd.read_csv(file, usecols=[" Label"])

    print("\n" + "=" * 80)
    print(file.name)
    print(df[" Label"].value_counts())


Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
 Label
DDoS      128027
BENIGN     97718
Name: count, dtype: int64

Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
 Label
PortScan    158930
BENIGN      127537
Name: count, dtype: int64

Friday-WorkingHours-Morning.pcap_ISCX.csv
 Label
BENIGN    189067
Bot         1966
Name: count, dtype: int64

Monday-WorkingHours.pcap_ISCX.csv
 Label
BENIGN    529918
Name: count, dtype: int64

Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv
 Label
BENIGN          288566
Infiltration        36
Name: count, dtype: int64

Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv
 Label
BENIGN                        168186
Web Attack � Brute Force        1507
Web Attack � XSS                 652
Web Attack � Sql Injection        21
Name: count, dtype: int64

Tuesday-WorkingHours.pcap_ISCX.csv
 Label
BENIGN         432074
FTP-Patator      7938
SSH-Patator      5897
Name: count, dtype: int64

Wednesday-workingHours.pcap_ISCX.csv
 Label
BENIGN

In [10]:
summary_df.to_csv(
    "../reports/dataset_summary.csv",
    index=False
)